In [2]:
from collections import Counter
from collections import defaultdict
import math
import numpy as np
import codecs
import tqdm
import time
import copy
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline
from rdkit import Chem


from utils import iter_smiles
import trie_funcs as tf
from ape_tokenizer import APETokenizer
from SmilesPE.tokenizer import SPE_Tokenizer        

In [7]:
# DeepChem ESOL / Delaney processed CSV
ESOL_URL = "http://deepchem.io.s3-website-us-west-1.amazonaws.com/datasets/delaney-processed.csv"

TARGET_COL = "measured log solubility in mols per litre"
SMILES_COL = "smiles"

def canonicalize_smiles(smiles: str) -> str | None:
    """Return RDKit canonical SMILES or None if parsing fails."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

def load_esol_canonical(path_or_url: str = ESOL_URL):
    """
    Download/load ESOL (Delaney) dataset, canonicalize SMILES,
    and return (smiles_list, y, full_df).

    smiles_list: list[str] of canonical SMILES
    y: np.ndarray of measured log solubility
    full_df: pandas DataFrame with original + canonical columns
    """
    df = pd.read_csv(path_or_url)

    # Sanity check: ensure required columns exist
    required = {SMILES_COL, TARGET_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in ESOL CSV: {missing}")

    # Canonicalize SMILES
    df["smiles_canonical"] = df[SMILES_COL].apply(canonicalize_smiles)

    # Drop rows where RDKit failed to parse
    df = df.dropna(subset=["smiles_canonical"]).reset_index(drop=True)

    just_smiles = df[['smiles_canonical']].copy()
    just_smiles.rename(columns={"smiles_canonical": "SMILES"}, inplace=True)
    print(type(just_smiles['SMILES'][0]))
    just_smiles.to_parquet("data/esol_canonical.parquet", index=False)

    # Extract features & target
    X_smiles = df["smiles_canonical"].tolist()
    y = df[TARGET_COL].values.astype(float)

    return X_smiles, y, df

X_smiles, y, df_esol = load_esol_canonical()
print(f"# molecules: {len(X_smiles)}")
print("Example canonical SMILES:", X_smiles[0])
print("Example target (log solubility):", y[0])


<class 'str'>
# molecules: 1128
Example canonical SMILES: N#CC(OC1OC(COC2OC(CO)C(O)C(O)C2O)C(O)C(O)C1O)c1ccccc1
Example target (log solubility): -0.77


In [8]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(X_smiles))
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=0)

X_train = [X_smiles[i] for i in train_idx]
X_test  = [X_smiles[i] for i in test_idx]

# and same for APE / SPE
y_train, y_test = y[train_idx], y[test_idx]

## Train Trie Tokenizer

In [9]:
# train_trie.py
SLICE = 'data/esol_canonical.parquet'

trie_state = tf.prepare_compressor(iter_smiles(SLICE), K=8, freq_thr=4)
tf.save_state(trie_state, 'exp6/trie.pkl')

In [10]:
X_trie_train = []
for s in X_train:
    toks = tf.compress_and_return(s, trie_state)
    X_trie_train.append(" ".join(toks))  # join tokens with spaces

X_trie_test = []
for s in X_test:
    toks = tf.compress_and_return(s, trie_state)
    X_trie_test.append(" ".join(toks))  # join tokens with spaces


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"[^ ]+",
    lowercase=False,
    max_features=5000,
)

X_trie_train_tfidf = vectorizer.fit_transform(X_trie_train)
X_trie_test_tfidf  = vectorizer.transform(X_trie_test)

print("Train TF-IDF:", X_trie_train_tfidf.shape)
print("Test  TF-IDF:", X_trie_test_tfidf.shape)


Train TF-IDF: (902, 1241)
Test  TF-IDF: (226, 1241)


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rf = RandomForestRegressor(
    n_estimators=500,
    n_jobs=-1,
    random_state=0,
)

rf.fit(X_trie_train_tfidf, y_train)
y_pred = rf.predict(X_trie_test_tfidf)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"Trie QSAR → RMSE={rmse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")


Trie QSAR → RMSE=1.638, MAE=1.243, R²=0.442


## TTG

In [13]:
# train_trie.py
SLICE = 'data/esol_canonical.parquet'

ttg_state = tf.prepare_compressor_with_ttg(iter_smiles(SLICE), K=10, freq_thr=3, entropy_thr=3.5)
tf.save_state(ttg_state, 'exp6/ttg.pkl')

In [14]:
X_ttg_train = []
for s in X_train:
    toks = tf.compress_and_return(s, ttg_state)
    X_ttg_train.append(" ".join(toks))  # join tokens with spaces

X_ttg_test = []
for s in X_test:
    toks = tf.compress_and_return(s, ttg_state)
    X_ttg_test.append(" ".join(toks))  # join tokens with spaces


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"[^ ]+",
    lowercase=False,
    max_features=5000,
)

X_ttg_train_tfidf = vectorizer.fit_transform(X_ttg_train)
X_ttg_test_tfidf  = vectorizer.transform(X_ttg_test)

print("Train TF-IDF:", X_ttg_train_tfidf.shape)
print("Test  TF-IDF:", X_ttg_test_tfidf.shape)


Train TF-IDF: (902, 1333)
Test  TF-IDF: (226, 1333)


In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rf = RandomForestRegressor(
    n_estimators=500,
    n_jobs=-1,
    random_state=0,
)

rf.fit(X_ttg_train_tfidf, y_train)
y_pred = rf.predict(X_ttg_test_tfidf)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"TTG QSAR → RMSE={rmse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")


TTG QSAR → RMSE=1.831, MAE=1.393, R²=0.302


## XGBoost

In [86]:
import xgboost as xgb

gb = xgb.XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",   
    n_jobs=-1,
)

gb.fit(X_trie_train_tfidf, y_train)
y_pred = gb.predict(X_trie_test_tfidf)


In [88]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"XGBoost Trie → RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")


XGBoost Trie → RMSE=1.6576, MAE=1.2747, R²=0.4284


In [84]:
import xgboost as xgb

gb = xgb.XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",   
    n_jobs=-1,
)

gb.fit(X_ttg_train_tfidf, y_train)
y_pred = gb.predict(X_ttg_test_tfidf)


In [85]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"XGBoost TTG → RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")


XGBoost TTG → RMSE=2.0139, MAE=1.5517, R²=0.1563


## APE

In [ ]:
# train_trie.py
SLICE = 'data/esol_canonical.parquet'
ape = APETokenizer()

ape.train(iter_smiles(SLICE), max_vocab_size=4890, min_freq_for_merge=800)
ape.save_pretrained('exp6/ape')

In [17]:
APE_DIR = "exp6/ape"
ape_state = APETokenizer.from_pretrained(APE_DIR)
ape_state.load_vocabulary("exp6/ape/vocab.json")


In [20]:
# tokenize dataset
X_ape_train = []
for s in X_train:
    ids = ape_state.encode(s)
    toks = ape_state.convert_ids_to_tokens(ids)
    X_ape_train.append(" ".join(toks))

X_ape_test = []
for s in X_test:
    ids = ape_state.encode(s)
    toks = ape_state.convert_ids_to_tokens(ids)
    X_ape_test.append(" ".join(toks))


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"[^ ]+",
    lowercase=False,
    max_features=5000,
)

X_ape_train_tfidf = vectorizer.fit_transform(X_ape_train)
X_ape_test_tfidf  = vectorizer.transform(X_ape_test)

print("Train TF-IDF:", X_ape_train_tfidf.shape)
print("Test  TF-IDF:", X_ape_test_tfidf.shape)


Train TF-IDF: (902, 1221)
Test  TF-IDF: (226, 1221)


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rf = RandomForestRegressor(
    n_estimators=500,
    n_jobs=-1,
    random_state=0,
)

rf.fit(X_ape_train_tfidf, y_train)
y_pred = rf.predict(X_ape_test_tfidf)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"APE QSAR → RMSE={rmse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")


APE QSAR → RMSE=2.025, MAE=1.569, R²=0.147
